# Streamer Initial Condition Estimator
Estimate V_los and other derived quantities from IC parameters before running the full integration.

In [1]:
import numpy as np
import sys
sys.path.insert(0, '.')

from streamer_ic import (
    get_streamer_initial_state,
    cartesian_to_spherical,
    get_axis_unit_vector,
    build_local_frame,
)
from streamer_model import gm_from_mstar

OMP: Warning #182: GOMP_STACKSIZE: ignored because KMP_STACKSIZE has been defined
OMP: Warning #182: OMP_STACKSIZE: ignored because KMP_STACKSIZE has been defined


In [7]:
# ============================================================
# Parameters — edit these
# ============================================================

M_star = 15.0
GM = gm_from_mstar(M_star)

model_type = 'mendoza'  # 'mendoza' or 'ulrich'

cart_x, cart_y, cart_z = -440, 1200, -1000
r0_init, theta0_init, phi0_init = cartesian_to_spherical(cart_x,cart_y,cart_z)

# --- Spherical ICs (model convention: theta from +y, phi from +z in XZ) ---
r0 = r0_init          # AU
theta_part_deg = theta0_init  # deg, from +y
phi_part_deg = phi0_init    # deg, from +z in XZ

# --- Rotation axis ---
theta_axis_deg = 30.0
phi_axis_deg = 180.0

# --- Model-specific ---
v_r = -2.0           # km/s, radial infall (Mendoza)
omega_round_yr = -10e-5  # round/yr (Mendoza)
Rc = 100.0           # AU, centrifugal radius (Ulrich)

# ============================================================

if model_type == 'mendoza':
    ic = get_streamer_initial_state('mendoza',
        r0=r0, theta_part_deg=theta_part_deg, phi_part_deg=phi_part_deg,
        v_r=v_r, omega_round_yr=omega_round_yr,
        theta_axis_deg=theta_axis_deg, phi_axis_deg=phi_axis_deg)
elif model_type == 'ulrich':
    ic = get_streamer_initial_state('ulrich',
        r0=r0, theta_part_deg=theta_part_deg, phi_part_deg=phi_part_deg,
        GM=GM, Rc=Rc,
        theta_axis_deg=theta_axis_deg, phi_axis_deg=phi_axis_deg)

x0, y0, z0, vx0, vy0, vz0 = ic
r_from_xyz = np.sqrt(x0**2 + y0**2 + z0**2)
v_mag = np.sqrt(vx0**2 + vy0**2 + vz0**2)

In [8]:
print('=' * 55)
print(f'Model: {model_type.upper()}')
print(f'GM = {GM:.1f} km²/s²·AU  (M = {M_star} M☉)')
print('=' * 55)
print()
print('--- Position ---')
print(f'  Cartesian:  ({x0:>10.1f}, {y0:>10.1f}, {z0:>10.1f}) AU')
print(f'  Radius:      {r_from_xyz:>10.1f} AU  (input: {r0})')
print()
print('--- Velocity ---')
print(f'  Cartesian:  ({vx0:>10.2f}, {vy0:>10.2f}, {vz0:>10.2f}) km/s')
print(f'  |v|:         {v_mag:>10.2f} km/s')
print()
print(f'>>> Initial V_los = vz0 = {vz0:.3f} km/s')
print()

# Derived quantities
v_radial = (vx0*x0 + vy0*y0 + vz0*z0) / r_from_xyz
v_tangential = np.sqrt(max(v_mag**2 - v_radial**2, 0))
print('--- Derived ---')
print(f'  Radial velocity:     {v_radial:>10.2f} km/s')
print(f'  Tangential velocity: {v_tangential:>10.2f} km/s')

# Angular momentum
L_vec = np.cross([x0, y0, z0], [vx0, vy0, vz0])
L_mag = np.linalg.norm(L_vec)
L_dir = L_vec / L_mag if L_mag > 1e-10 else np.array([0, 0, 1])
print(f'  |L| (specific):      {L_mag:>10.1f} AU·km/s')
print(f'  L direction:         ({L_dir[0]:.3f}, {L_dir[1]:.3f}, {L_dir[2]:.3f})')

# Free-fall timescale
t_ff = np.pi * np.sqrt(r_from_xyz**3 / (8 * GM))
print(f'  Free-fall timescale: {t_ff:>10.1f} (dimensionless time units)')
print(f'                        ≈ {t_ff * 4.74:.0f} yr')

Model: MENDOZA
GM = 13306.9 km²/s²·AU  (M = 15.0 M☉)

--- Position ---
  Cartesian:  (    -440.0,     1200.0,    -1000.0) AU
  Radius:          1622.8 AU  (input: 1622.8370220080635)

--- Velocity ---
  Cartesian:  (      1.33,      -2.13,       0.10) km/s
  |v|:               2.52 km/s

>>> Initial V_los = vz0 = 0.097 km/s

--- Derived ---
  Radial velocity:          -2.00 km/s
  Tangential velocity:       1.53 km/s
  |L| (specific):          2485.3 AU·km/s
  L direction:         (-0.812, -0.520, -0.267)
  Free-fall timescale:      629.5 (dimensionless time units)
                        ≈ 2984 yr


In [4]:
# Quick 3D preview of initial state
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=('PPP', 'PPV'),
    horizontal_spacing=0.08,
)

# Left: XYZ
fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode='markers',
    marker=dict(size=6, color='black', symbol='diamond'), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter3d(x=[x0], y=[y0], z=[z0], mode='markers',
    marker=dict(size=6, color='yellow', symbol='square'), showlegend=False), row=1, col=1)

# Right: XY-V_los
fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode='markers',
    marker=dict(size=6, color='black', symbol='diamond'), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter3d(x=[x0], y=[y0], z=[vz0], mode='markers',
    marker=dict(size=6, color='yellow', symbol='square'), showlegend=False), row=1, col=2)

camera = dict(up=dict(x=0,y=1,z=0), center=dict(x=0,y=0,z=0), eye=dict(x=0,y=0,z=-2.5))
fig.update_layout(template='plotly_white', showlegend=False,
    scene1=dict(xaxis=dict(title='X'), yaxis=dict(title='Y'), zaxis=dict(title='Z'), aspectmode='data'),
    scene2=dict(xaxis=dict(title='X'), yaxis=dict(title='Y'), zaxis=dict(title='V_los', range=[-5,5]), aspectmode='data'),
    scene_camera=camera)
fig.show()